# DEFT vs DEFT_perf 




In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd 

from src.data.polymerase_loader import load_polymerase
from src.utils.lightweight_tree import load_tree_root
from src.utils.tree import get_results

def _find_repo_root(start: Path | None = None) -> Path:
    cur = (start or Path.cwd()).resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'conf').exists():
            return candidate
    raise FileNotFoundError('Could not locate repository root from current working directory.')

BASE_DEFT = _find_repo_root()

SEEDS = [42, 43, 44, 45, 46]
MAX_DEPTH = 3
TEST_SIZE = 0.2
PATH_SAVE_DEFT_PERF_RAW = BASE_DEFT / 'results' / 'polymerase' / 'results_deft_perf_recomputed_from_trees.csv'
PATH_SAVE_DEFT_PERF_SUMMARY = BASE_DEFT / 'results' / 'polymerase' / 'results_deft_perf_summary_depth1_3.csv'
PATH_SAVE_TRADEOFF_TABLE = BASE_DEFT / 'results' / 'polymerase' / 'results_deft_vs_deft_perf_tradeoff_depth1_3.csv'

DEFT_TREE_PATHS = {
    42: BASE_DEFT / 'artifacts/trees/polymerase/deft_lightweight/tree_001_seed42.dill',
    43: BASE_DEFT / 'artifacts/trees/polymerase/deft_lightweight/tree_002_seed43.dill',
    44: BASE_DEFT / 'artifacts/trees/polymerase/deft_lightweight/tree_003_seed44.dill',
    45: BASE_DEFT / 'artifacts/trees/polymerase/deft_lightweight/tree_004_seed45.dill',
    46: BASE_DEFT / 'artifacts/trees/polymerase/deft_lightweight/tree_005_seed46.dill',
}

DEFT_PERF_TREE_PATHS = {
    42: BASE_DEFT / 'artifacts/trees/polymerase/deft_perf_lightweight/tree_002_seed42.pkl',
    43: BASE_DEFT / 'artifacts/trees/polymerase/deft_perf_lightweight/tree_003_seed43.pkl',
    44: BASE_DEFT / 'artifacts/trees/polymerase/deft_perf_lightweight/tree_004_seed44.pkl',
    45: BASE_DEFT / 'artifacts/trees/polymerase/deft_perf_lightweight/tree_005_seed45.pkl',
    46: BASE_DEFT / 'artifacts/trees/polymerase/deft_perf_lightweight/tree_006_seed46.pkl',
}


In [13]:
def _flatten_y(y):
    return np.asarray(y).reshape(-1)


def load_polymerase_eval_data(seed: int, test_size: float = 0.2):
    X_train, X_test, y_train, y_test = load_polymerase(
        use_raw_sequence=True,
        type_dataset='Plus',
        no_prior_knowledge=False,
        test_size=test_size,
        random_state=seed,
        use_external_test_set=True,
    )
    return X_train, X_test, _flatten_y(y_train), _flatten_y(y_test)


def runs_from_hardcoded_paths(path_map: dict[int, Path]) -> pd.DataFrame:
    rows = []
    missing = []

    for seed in SEEDS:
        if seed not in path_map:
            missing.append(f'seed {seed}: missing from mapping')
            continue

        p = Path(path_map[seed])
        if not p.exists():
            missing.append(f'seed {seed}: file not found -> {p}')
            continue

        rows.append({'seed': int(seed), 'tree_path': str(p)})

    if missing:
        msg = 'Missing hardcoded tree paths:\\n' + '\\n'.join(missing)
        raise RuntimeError(msg)

    return pd.DataFrame(rows).sort_values('seed').reset_index(drop=True)


def evaluate_from_artifacts(
    runs_df: pd.DataFrame,
    method_name: str,
    max_depth: int = 3,
    test_size: float = 0.2,
) -> pd.DataFrame:
    cols = ['depth', 'accuracy', 'auprc', 'f1', 'precision', 'recall', 'isTrain', 'method', 'random_seed']
    if runs_df.empty:
        return pd.DataFrame(columns=cols)

    results = []

    for _, row in runs_df.iterrows():
        seed = int(row['seed'])
        root = load_tree_root(Path(row['tree_path']), install_legacy_shims=True)
        X_train, X_test, y_train, y_test = load_polymerase_eval_data(
            seed=seed,
            test_size=test_size,
        )
        df_seed = get_results(
            root=root,
            X_train=X_train,
            X_test=X_test,
            y_train=y_train,
            y_test=y_test,
            range_depth=range(1, max_depth + 1),
            name_method=method_name,
            random_seed=seed,
        )
        df_seed['random_seed'] = df_seed['random_seed'].astype(int)
        results.append(df_seed[cols])

    return pd.concat(results, ignore_index=True)



def summarize_accuracy(df: pd.DataFrame) -> pd.DataFrame:
    out = (
        df.groupby(['method', 'depth', 'isTrain'])['accuracy']
        .agg(mean='mean', std='std')
        .reset_index()
    )
    return out


def build_tradeoff_table(summary: pd.DataFrame, method_deft: str = 'DEFT', method_perf: str = 'DEFT_perf') -> pd.DataFrame:
    rows = []

    for depth in sorted(summary['depth'].unique()):
        def pick(method, is_train):
            sel = summary[(summary['method'] == method) & (summary['depth'] == depth) & (summary['isTrain'] == is_train)]
            if sel.empty:
                return np.nan, np.nan
            return float(sel['mean'].iloc[0]), float(sel['std'].iloc[0])

        dt_m, dt_s = pick(method_deft, True)
        dp_m, dp_s = pick(method_perf, True)
        dte_m, dte_s = pick(method_deft, False)
        dpe_m, dpe_s = pick(method_perf, False)

        rows.append(
            {
                'depth': int(depth),
                'DEFT_train_mean': dt_m,
                'DEFT_train_std': dt_s,
                'DEFT_perf_train_mean': dp_m,
                'DEFT_perf_train_std': dp_s,
                'DEFT_test_mean': dte_m,
                'DEFT_test_std': dte_s,
                'DEFT_perf_test_mean': dpe_m,
                'DEFT_perf_test_std': dpe_s,
            }
        )

    table = pd.DataFrame(rows).sort_values('depth').reset_index(drop=True)

    def fmt(m, s):
        if pd.isna(m) or pd.isna(s):
            return 'NA'
        return f'{m:.3f} ({s:.3f})'

    table_fmt = table.copy()
    table_fmt['DEFT_train'] = [fmt(m, s) for m, s in zip(table['DEFT_train_mean'], table['DEFT_train_std'])]
    table_fmt['DEFT_perf_train'] = [fmt(m, s) for m, s in zip(table['DEFT_perf_train_mean'], table['DEFT_perf_train_std'])]
    table_fmt['DEFT_test'] = [fmt(m, s) for m, s in zip(table['DEFT_test_mean'], table['DEFT_test_std'])]
    table_fmt['DEFT_perf_test'] = [fmt(m, s) for m, s in zip(table['DEFT_perf_test_mean'], table['DEFT_perf_test_std'])]

    cols_fmt = ['depth', 'DEFT_train', 'DEFT_perf_train', 'DEFT_test', 'DEFT_perf_test']
    return table, table_fmt[cols_fmt] 


In [14]:
original_runs = runs_from_hardcoded_paths(DEFT_TREE_PATHS)
eoh_runs = runs_from_hardcoded_paths(DEFT_PERF_TREE_PATHS)

display(original_runs)

display(eoh_runs)


,seed,tree_path
0,42,/home/nvth2/deft_cr/artifacts/trees/polymerase...
1,43,/home/nvth2/deft_cr/artifacts/trees/polymerase...
2,44,/home/nvth2/deft_cr/artifacts/trees/polymerase...
3,45,/home/nvth2/deft_cr/artifacts/trees/polymerase...
4,46,/home/nvth2/deft_cr/artifacts/trees/polymerase...


,seed,tree_path
0,42,/home/nvth2/deft_cr/artifacts/trees/polymerase...
1,43,/home/nvth2/deft_cr/artifacts/trees/polymerase...
2,44,/home/nvth2/deft_cr/artifacts/trees/polymerase...
3,45,/home/nvth2/deft_cr/artifacts/trees/polymerase...
4,46,/home/nvth2/deft_cr/artifacts/trees/polymerase...


In [15]:
df_deft = evaluate_from_artifacts(
    runs_df=original_runs,
    method_name='DEFT',
    max_depth=MAX_DEPTH,
    test_size=TEST_SIZE,
)

df_deft_perf = evaluate_from_artifacts(
    runs_df=eoh_runs,
    method_name='DEFT_perf',
    max_depth=MAX_DEPTH,
    test_size=TEST_SIZE,
)

df_all = pd.concat([df_deft, df_deft_perf], ignore_index=True)
df_all['random_seed'] = df_all['random_seed'].astype(int)
df_all['depth'] = df_all['depth'].astype(int)
df_all['isTrain'] = df_all['isTrain'].astype(bool)

summary = summarize_accuracy(df_all)
tradeoff_numeric, tradeoff_formatted = build_tradeoff_table(summary)

print('Reconstructed table (mean (std), accuracy):')
display(tradeoff_formatted)


# Persist DEFT_perf outputs so they can be reused without rerunning tree evaluation
for out_path in [PATH_SAVE_DEFT_PERF_RAW, PATH_SAVE_DEFT_PERF_SUMMARY, PATH_SAVE_TRADEOFF_TABLE]:
    out_path.parent.mkdir(parents=True, exist_ok=True)

df_deft_perf.to_csv(PATH_SAVE_DEFT_PERF_RAW, index=False)
summary[summary['method'] == 'DEFT_perf'].to_csv(PATH_SAVE_DEFT_PERF_SUMMARY, index=False)
tradeoff_numeric.to_csv(PATH_SAVE_TRADEOFF_TABLE, index=False)

print(f'Saved DEFT_perf raw results to: {PATH_SAVE_DEFT_PERF_RAW}')
print(f'Saved DEFT_perf summary to: {PATH_SAVE_DEFT_PERF_SUMMARY}')
print(f'Saved DEFT vs DEFT_perf tradeoff table to: {PATH_SAVE_TRADEOFF_TABLE}')


Reconstructed table (mean (std), accuracy):


,depth,DEFT_train,DEFT_perf_train,DEFT_test,DEFT_perf_test
0,1,0.769 (0.004),0.820 (0.012),0.762 (0.004),0.810 (0.012)
1,2,0.836 (0.017),0.872 (0.020),0.823 (0.024),0.855 (0.014)
2,3,0.872 (0.009),0.893 (0.006),0.856 (0.014),0.867 (0.007)


Saved DEFT_perf raw results to: /home/nvth2/deft_cr/results/polymerase/results_deft_perf_recomputed_from_trees.csv
Saved DEFT_perf summary to: /home/nvth2/deft_cr/results/polymerase/results_deft_perf_summary_depth1_3.csv
Saved DEFT vs DEFT_perf tradeoff table to: /home/nvth2/deft_cr/results/polymerase/results_deft_vs_deft_perf_tradeoff_depth1_3.csv
